# Matching Networks — Matching Networks for One Shot Learning (2016)

_Papers · Meta-learning, Bayesian & Robustness_

**Classify a new example by an attention-weighted vote over a tiny labelled support set — and train the model the exact same way you test it.**

---

This notebook is a practice scaffold for the **Matching Networks — Matching Networks for One Shot Learning (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the worked example: cosine -> softmax -> weighted label vote. ---
fq = torch.tensor([1.0, 0.5])
g  = torch.tensor([[1.0, 0.4], [-0.5, 1.0], [0.2, -1.0]])   # one support example per class
Y  = torch.eye(3)                                            # one-hot support labels
cos  = F.cosine_similarity(fq.unsqueeze(0), g, dim=1)        # c(f(x_hat), g(x_i))
a    = torch.softmax(cos, dim=0)                             # attention weights (sum to 1)
yhat = (a.unsqueeze(1) * Y).sum(0)                           # Eqn. 1: sum_i a_i * y_i
print("worked cos :", [round(v, 4) for v in cos.tolist()])   # [0.9965, 0.0, -0.2631]
print("worked a   :", [round(v, 4) for v in a.tolist()])     # [0.605, 0.2233, 0.1717]
print("worked yhat:", [round(v, 4) for v in yhat.tolist()], " -> class", int(yhat.argmax()))


# --- 1. A hard synthetic few-shot task: classes live in a hidden subspace + heavy noise. ---
NUM_CLASSES, DIM, HID = 60, 64, 8
torch.manual_seed(3)
W           = torch.randn(DIM, HID)                          # the hidden "true" subspace
codes       = torch.randn(NUM_CLASSES, HID) * 2.0
class_means = codes @ W.T
class_means = class_means / class_means.norm(dim=1, keepdim=True) * 3.0
def sample_pts(cls, m):
    return class_means[cls] + torch.randn(m, DIM) * 1.2      # noise so raw cosine ~ chance


# --- 2. One shared embedding network (used for both f and g). ---
class Encoder(nn.Module):
    def __init__(self, d_in=DIM, d=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Linear(128, d))
    def forward(self, x):
        return self.net(x)

class Identity(nn.Module):                                   # the ablation "encoder"
    def forward(self, x):
        return x


# --- 3. Sample one N-way K-shot episode (train protocol == test protocol). ---
def episode(N=5, K=1, Q=5):
    classes = np.random.choice(NUM_CLASSES, N, replace=False)
    sx, sy, qx, qy = [], [], [], []
    for local_label, c in enumerate(classes):               # relabel chosen classes 0..N-1
        pts = sample_pts(c, K + Q)
        sx.append(pts[:K]); sy += [local_label] * K
        qx.append(pts[K:]); qy += [local_label] * Q
    return torch.cat(sx), torch.tensor(sy), torch.cat(qx), torch.tensor(qy), N


# --- 4. The Matching Networks classifier: cosine-softmax attention + weighted label vote. ---
def log_probs(enc, sx, sy, qx, N):
    fs, gs = enc(qx), enc(sx)                                # query and support embeddings
    cos = F.cosine_similarity(fs.unsqueeze(1), gs.unsqueeze(0), dim=2)  # [n_query, n_support]
    a   = torch.softmax(cos, dim=1)                          # Section 2.1.1: softmax over support
    Y   = F.one_hot(sy, N).float()                           # [n_support, N]
    yhat = a @ Y                                             # Eqn. 1: sum_i a(x_hat,x_i) y_i
    return torch.log(yhat + 1e-8)                            # log for NLL loss

def evaluate(enc, N=5, K=1, episodes=300):
    enc.eval(); correct = total = 0
    with torch.no_grad():
        for _ in range(episodes):
            sx, sy, qx, qy, n = episode(N, K)
            pred = log_probs(enc, sx, sy, qx, n).argmax(1)
            correct += (pred == qy).sum().item(); total += len(qy)
    return correct / total


# --- 5. Train episodically; evaluate on held-out class draws; ablate the encoder. ---
raw = Identity()
acc_raw = evaluate(raw, 5, 1)                                # ablation: no learned metric

enc = Encoder()
opt = torch.optim.Adam(enc.parameters(), lr=1e-3)
acc_before = evaluate(enc, 5, 1)
for step in range(800):
    enc.train()
    sx, sy, qx, qy, n = episode(N=5, K=1, Q=5)               # a fresh episode every step
    loss = F.nll_loss(log_probs(enc, sx, sy, qx, n), qy)
    opt.zero_grad(); loss.backward(); opt.step()

print("\nchance (5-way) = 0.20")
print("raw-input cosine, no learning (ABLATION):", round(acc_raw, 4))
print("learned encoder, untrained             :", round(acc_before, 4))
print("learned encoder, 5-way 1-shot          :", round(evaluate(enc, 5, 1), 4))
print("learned encoder, 5-way 5-shot          :", round(evaluate(enc, 5, 5), 4))
print("learned encoder, 20-way 1-shot         :", round(evaluate(enc, 20, 1), 4), "(chance 0.05)")
# Trained accuracy lands well above chance; more shots (5 vs 1) helps; 20-way stays far above 0.05.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does episodic cosine-softmax training lift held-out 5-way 1-shot accuracy above the 20% chance line, and does a learned embedding beat raw-input cosine?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

# Reproduces the qualitative effect on toy data: episodic cosine-softmax training
# lifts held-out N-way K-shot accuracy well above 1/N chance; a learned embedding
# beats raw-input cosine. (Full trainable version is in the CODE cell above.)
torch.manual_seed(0); np.random.seed(0)
NUM_CLASSES, DIM, HID = 60, 64, 8
torch.manual_seed(3)
W = torch.randn(DIM, HID); codes = torch.randn(NUM_CLASSES, HID) * 2.0
class_means = codes @ W.T
class_means = class_means / class_means.norm(dim=1, keepdim=True) * 3.0
sample = lambda c, m: class_means[c] + torch.randn(m, DIM) * 1.2

class Encoder(nn.Module):
    def __init__(s, d=32):
        super().__init__(); s.net = nn.Sequential(nn.Linear(DIM,128), nn.ReLU(), nn.Linear(128,d))
    def forward(s, x): return s.net(x)

def episode(N=5, K=1, Q=5):
    cs = np.random.choice(NUM_CLASSES, N, replace=False); sx,sy,qx,qy=[],[],[],[]
    for lab,c in enumerate(cs):
        p = sample(c, K+Q); sx.append(p[:K]); sy+=[lab]*K; qx.append(p[K:]); qy+=[lab]*Q
    return torch.cat(sx), torch.tensor(sy), torch.cat(qx), torch.tensor(qy), N

def lp(enc, sx, sy, qx, N):
    cos = F.cosine_similarity(enc(qx).unsqueeze(1), enc(sx).unsqueeze(0), dim=2)
    return torch.log(torch.softmax(cos,1) @ F.one_hot(sy,N).float() + 1e-8)

def ev(enc, N=5, K=1, ep=120):
    enc.eval(); c=t=0
    with torch.no_grad():
        for _ in range(ep):
            sx,sy,qx,qy,n = episode(N,K); c+=(lp(enc,sx,sy,qx,n).argmax(1)==qy).sum().item(); t+=len(qy)
    return c/t

enc = Encoder(); opt = torch.optim.Adam(enc.parameters(), lr=1e-3); curve=[]
for step in range(800):
    enc.train(); sx,sy,qx,qy,n = episode(5,1,5)
    loss = F.nll_loss(lp(enc,sx,sy,qx,n), qy); opt.zero_grad(); loss.backward(); opt.step()
    if step % 25 == 0: curve.append([step, round(ev(enc,5,1), 4)])
print("accuracy curve (5-way 1-shot, chance 0.20):", curve)
print("trained 5w5s:", round(ev(enc,5,5),4), " trained 20w1s:", round(ev(enc,20,1),4), "(chance 0.05)")
# Curve climbs from ~0.25 to ~0.55; more shots help; 20-way 1-shot stays far above 0.05.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working episodic Matching Network whose 5-way 1-shot accuracy is
            well above the 20% chance line. Replace the learned embedding enc(x) with the
            raw input (an identity "encoder"), keeping the cosine-softmax vote and the episode sampler
            identical, and re-evaluate. What happens to accuracy, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the encoder: replace enc with lambda x: x so cosine similarity is taken in the raw input space; keep the attention vote, episodes, and data identical. — _An honest ablation changes exactly one thing &mdash; the learned embedding &mdash; so any drop is attributable to it._
- Re-evaluate on held-out classes at 5-way 1-shot and compare to the learned-embedding number and to 20% chance. — _Raw-input cosine sits near chance because same-class points are not aligned until the network learns an embedding that aligns them._
- Conclude that the lift comes from the learned metric, not from the attention formula alone. — _The attention vote is fixed in both runs; only the embedding differs, isolating it as the cause._

**Answer:** Accuracy drops sharply toward the 20% chance line. The cosine-softmax vote is unchanged, so
                 the only thing lost is the learned embedding. This shows Matching Networks works by
                 learning a metric in which same-class points align &mdash; in our run, raw-input cosine scored
                 about 0.39 (5-way) while the trained encoder reached about 0.53. The attention formula is just
                 the differentiable nearest-neighbour rule on top of that learned space.

</details>

**Problem 2.** Redo the worked example but suppose the query embeds to $f(\hat{x})=[0.2,\,-1.0]$ (pointing the
            same way as the class-2 support example). With the same three support embeddings, which class wins,
            and roughly how confident is the vote?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute cosine similarities to each support: now $c_2$ (with $g(x_2)=[0.2,-1.0]$) is $\approx 1$, while $c_0$ and $c_1$ are smaller (the query now points down-right). — _Cosine is largest for the support example the query is most aligned with &mdash; here class 2._
- Softmax over the three scores: the large $c_2$ exponentiates to dominate, so $a_2$ is the biggest weight. — _Softmax turns the highest similarity into the loudest vote._
- Apply Eqn. 1: $\hat{y}=\sum_i a_i y_i$ is largest in slot 2, so the predicted class is 2. — _The weighted one-hot sum just returns the dominant weight's class._

**Answer:** Class 2 wins. The query now aligns with $g(x_2)$, so $c_2\approx 1$ is the largest
                 cosine, the softmax concentrates weight on $a_2$, and Eqn. 1's weighted vote peaks at slot 2.
                 The vote is fairly confident because one cosine is near $+1$ while the others are much smaller,
                 so the softmax is sharply peaked &mdash; the soft nearest-neighbour rule lands on the aligned
                 support example's class.

</details>

**Problem 3.** You train on ordinary fixed-class mini-batches (a normal 10-class classifier) and then test it as a
            5-way 1-shot Matching Network on brand-new classes. The accuracy is near chance. Using Section 2.2,
            explain why, and what to change.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the mismatch: training optimized a fixed-class classifier, but the test asks for a support-set vote over unseen classes. — _Section 2.2: "The training procedure has to be chosen carefully so as to match inference at test time." A fixed-class objective does not exercise the support-set mechanism._
- Switch to episodic training: each step, sample 5 classes and 1 support example each, predict the queries with the cosine-softmax vote, and backprop that loss. — _Now every training step is itself a 5-way 1-shot problem, so the embedding is optimized for exactly the test-time vote._
- Hold out the test classes from training and evaluate at the same N-way K-shot setting. — _Matching train and test protocols (and class splits) is what makes the few-shot accuracy meaningful._

**Answer:** The model never practised the support-set vote during training, so it never learned an
                 embedding that makes the cosine-softmax classifier work &mdash; a violation of the paper's
                 "match inference at test time" principle (Section 2.2). Fix it by training episodically:
                 sample N-way K-shot episodes as the training objective, on classes disjoint from the test
                 classes. Then the train protocol equals the test protocol and accuracy rises well above
                 chance.

</details>